# 머신러닝 기초 0장. 준비

사회약학 연구방법 노트 '머신러닝 기초' 0장을 따라 하는 노트북입니다. 과목 전체가 쓰는 가상 청구자료(40세 이상 15,000명)를 불러와 학습·시험 자료로 나누고, 전처리와 모형을 파이프라인으로 묶고, 교차검증으로 조절값을 고른 뒤 분류(2023년 응급 입원)와 회귀(2023년 의료비)의 성능 지표를 구합니다. 셀을 위에서부터 차례로 실행하세요. 자료는 사이트가 만든 가상 자료이며 실제 환자 자료가 아닙니다.


## 가. 지도학습이란

자료를 불러와 특성(X)과 목표(y)를 정하고 fit → predict를 한 번 해 봅니다.

### 셀 1. 공통 자료 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
df = pd.read_csv(BASE + "ml_claims.csv")
print(df.shape)
df.head(3).T                     # 앞 3명을 세로로 돌려 보기

### 셀 2. 결과 두 개의 분포

In [ ]:
print(df["admit_2023"].value_counts())
print("admission rate:", round(df["admit_2023"].mean(), 4))
print(df["cost_2023"].describe().round(1))
print("zero cost:", round((df["cost_2023"] == 0).mean(), 3))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.hist(df["cost_2023"], bins=np.arange(0, 3001, 50))
ax.set_xlabel("cost_2023 (10,000 KRW), shown up to 3,000")
ax.set_ylabel("Number of people")
plt.tight_layout()

### 셀 3. 결측이 있는 열과 나이별 검진 결측

In [ ]:
miss = df.isna().mean().round(3)          # 열마다 결측 비율
print(miss[miss > 0])

age_grp = pd.cut(df["age"], [39, 64, 79, 99])
print(df["bmi"].isna().groupby(age_grp, observed=True)
      .mean().round(3))

### 셀 4. 특성 X와 목표 y

In [ ]:
y = df["admit_2023"]                                  # 목표
X = df.drop(columns=["id", "admit_2023", "cost_2023"])  # 특성
print(X.shape, y.shape)
print(X.dtypes.value_counts())

### 셀 5. fit과 predict

In [ ]:
from sklearn.linear_model import LogisticRegression

cols3 = ["age", "n_ed", "n_drugs"]       # 결측이 없는 특성 3개
model = LogisticRegression()             # (1) 모형을 고른다
model.fit(X[cols3], y)                   # (2) 정답이 있는 자료로 배운다

new = pd.DataFrame({"age": [55, 85], "n_ed": [0, 2],
                    "n_drugs": [3, 11]})  # 새로운 두 사람
print(model.predict(new))                # (3) 예측: 0 또는 1
print(model.predict_proba(new).round(3)) #     예측: 확률

## 나. 자료 나누기

이 과목의 모든 장이 셀 6의 분할을 그대로 씁니다.

### 셀 6. 이 과목의 분할 (모든 장이 같은 분할)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=2026)
print(X_train.shape, X_test.shape)
print("events:", y_train.sum(), y_test.sum())
print("rate  :", round(y_train.mean(), 4), round(y_test.mean(), 4))

### 셀 7. stratify 없이 나누면

In [ ]:
for seed in [1, 2, 3, 4, 5]:
    _, _, _, yt = train_test_split(X, y, test_size=0.25,
                                   random_state=seed)
    print("seed", seed, "events in test:", yt.sum(),
          " rate:", round(yt.mean(), 4))

### 셀 8. 배운 자료로 시험하면

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

cols = ["age", "female", "n_outpt", "n_ed", "n_admit",
        "n_drugs", "cost_2022"]
tree = DecisionTreeClassifier(random_state=0)   # 깊이 제한 없음
tree.fit(X_train[cols], y_train)

p_tr = tree.predict_proba(X_train[cols])[:, 1]
p_te = tree.predict_proba(X_test[cols])[:, 1]
print("train AUC:", round(roc_auc_score(y_train, p_tr), 3))
print("test AUC :", round(roc_auc_score(y_test, p_te), 3))
print("leaves:", tree.get_n_leaves())

## 다. 전처리와 파이프라인

결측 대체, 원-핫 인코딩, 표준화를 학습 자료로만 맞추고 모형과 묶습니다.

### 셀 9. 숫자 열과 범주 열

In [ ]:
cat_cols = ["insurance", "region", "smoking", "alcohol"]
num_cols = [c for c in X.columns if c not in cat_cols]
print(len(num_cols), "numeric,", len(cat_cols), "categorical")
X_train[cat_cols].head(4)

### 셀 10. 결측 대체 (SimpleImputer)

In [ ]:
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy="median")
imp.fit(X_train[["bmi", "egfr"]])        # 학습 자료에서 중앙값 계산
print("medians:", imp.statistics_)

four = X_train[["bmi", "egfr"]].iloc[:4]
print(four)
print(imp.transform(four))               # 결측을 중앙값으로 채움

### 셀 11. 원-핫 인코딩 (OneHotEncoder)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

smk = X_train[["smoking"]].fillna("missing")   # 결측도 한 범주로
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(smk)
print(ohe.get_feature_names_out())
print(smk.head(4))
print(ohe.transform(smk.head(4)))

### 셀 12. 표준화 (StandardScaler)

In [ ]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler().fit(X_train[["age", "n_outpt"]])
print("mean:", sc.mean_.round(2), " SD:", sc.scale_.round(2))
three = X_train[["age", "n_outpt"]].iloc[:3]
print(three)
print(sc.transform(three).round(2))

### 셀 13. ColumnTransformer와 Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def make_pipe(model, num=num_cols, cat=cat_cols):
    """전처리(결측 대체, 표준화, 원-핫)와 모형을 묶습니다."""
    num_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())])
    cat_steps = Pipeline([
        ("impute", SimpleImputer(strategy="constant",
                                 fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore",
                                 sparse_output=False))])
    prep = ColumnTransformer([("num", num_steps, num),
                              ("cat", cat_steps, cat)])
    return Pipeline([("prep", prep), ("model", model)])

pipe = make_pipe(LogisticRegression(max_iter=1000))
pipe.fit(X_train, y_train)
Z = pipe.named_steps["prep"].transform(X_test)
print("after prep:", Z.shape)
p_test = pipe.predict_proba(X_test)[:, 1]
print("test AUC:", round(roc_auc_score(y_test, p_test), 3))

### 셀 14. 표준화하지 않으면

In [ ]:
raw = Pipeline([("impute", SimpleImputer(strategy="median")),
                ("model", LogisticRegression())])   # 표준화 없음
raw.fit(X_train[num_cols], y_train)
print("iterations, no scaling:", raw.named_steps["model"].n_iter_)
print("iterations, scaled    :", pipe.named_steps["model"].n_iter_)

### 셀 15. 자료 누수: 결과 뒤의 정보가 섞이면

In [ ]:
rng = np.random.default_rng(0)
# 2023년(결과 기간)의 응급실 방문 수를 흉내 낸 열. 실제로는 쓰면 안 됨
ed_2023 = np.where(y == 1, rng.poisson(1.5, len(y)) + 1,
                   rng.poisson(0.08, len(y)))
X_leak = X.assign(n_ed_2023=ed_2023)

XL_train, XL_test = train_test_split(
    X_leak, test_size=0.25, stratify=y, random_state=2026)
leaky = make_pipe(LogisticRegression(max_iter=1000),
                  num=num_cols + ["n_ed_2023"])
leaky.fit(XL_train, y_train)
p_leak = leaky.predict_proba(XL_test)[:, 1]
print("test AUC with n_ed_2023:", round(roc_auc_score(y_test, p_leak), 3))

## 라. 교차검증과 조절값 탐색

조절값은 학습 자료 안의 교차검증으로 고르고, 시험 자료는 마지막에 한 번만 씁니다.

### 셀 16. 5겹으로 나누기 (StratifiedKFold)

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
for k, (tr, va) in enumerate(skf.split(X_train, y_train), start=1):
    print("fold", k, " train:", len(tr), " valid:", len(va),
          " events in valid:", y_train.iloc[va].sum())

### 셀 17. 교차검증 AUC (cross_val_score)

In [ ]:
from sklearn.model_selection import cross_val_score

auc5 = cross_val_score(make_pipe(LogisticRegression(max_iter=1000)),
                       X_train, y_train, cv=skf, scoring="roc_auc")
print(auc5.round(3))
print("mean:", round(auc5.mean(), 3), " SD:", round(auc5.std(), 3))

### 셀 18. 조절값 C 고르기 (GridSearchCV)

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {"model__C": [0.001, 0.01, 0.1, 1, 10, 100]}
gs = GridSearchCV(make_pipe(LogisticRegression(max_iter=1000)),
                  grid, cv=skf, scoring="roc_auc")
gs.fit(X_train, y_train)
print("best:", gs.best_params_, round(gs.best_score_, 4))

res = pd.DataFrame(gs.cv_results_)
show = ["param_model__C", "mean_test_score", "std_test_score",
        "rank_test_score"]
res[show].round(4)

### 셀 19. 시험 자료로 한 번 평가

In [ ]:
final = gs.best_estimator_      # 고른 C로 학습 자료 전체에 다시 맞춘 모형
p_final = final.predict_proba(X_test)[:, 1]
print("test AUC:", round(roc_auc_score(y_test, p_final), 3))

## 마. 성능을 재는 지표

분류는 정확도, 혼동행렬, 정밀도·재현율·F1, ROC와 정밀도-재현율 곡선, 회귀는 MAE, RMSE, R²를 봅니다.

### 셀 20. 정확도의 함정

In [ ]:
from sklearn.metrics import accuracy_score

all0 = np.zeros(len(y_test), dtype=int)     # 모두 '입원 안 함'
pred05 = final.predict(X_test)              # 임계값 0.5
print("accuracy, all 0  :", round(accuracy_score(y_test, all0), 4))
print("accuracy, model  :", round(accuracy_score(y_test, pred05), 4))
print("predicted 1 (0.5):", pred05.sum())

### 셀 21. 혼동행렬과 정밀도·재현율·F1

In [ ]:
from sklearn.metrics import (confusion_matrix, precision_score,
                             recall_score, f1_score)

pred10 = (p_final >= 0.10).astype(int)      # 임계값 0.10
print(confusion_matrix(y_test, pred10))
print("precision:", round(precision_score(y_test, pred10), 3))
print("recall   :", round(recall_score(y_test, pred10), 3))
print("F1       :", round(f1_score(y_test, pred10), 3))

### 셀 22. ROC 곡선과 정밀도-재현율 곡선

In [ ]:
from sklearn.metrics import (roc_curve, precision_recall_curve,
                             average_precision_score)

fpr, tpr, _ = roc_curve(y_test, p_final)
prec, rec, _ = precision_recall_curve(y_test, p_final)
print("AUC:", round(roc_auc_score(y_test, p_final), 3),
      " AP:", round(average_precision_score(y_test, p_final), 3))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
ax[0].plot(fpr, tpr)
ax[0].plot([0, 1], [0, 1], "--", color="gray")
ax[0].set_xlabel("1 - specificity")
ax[0].set_ylabel("Sensitivity (recall)")
ax[0].set_title("ROC curve")
ax[1].plot(rec, prec)
ax[1].axhline(y_test.mean(), ls="--", color="gray")
ax[1].set_xlabel("Recall")
ax[1].set_ylabel("Precision")
ax[1].set_title("Precision-recall curve")
plt.tight_layout()

### 셀 23. 회귀: 2023년 의료비 예측

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error,
                             mean_squared_error, r2_score)

c_train = df.loc[X_train.index, "cost_2023"]    # 같은 사람들의 비용
c_test = df.loc[X_test.index, "cost_2023"]
reg = make_pipe(LinearRegression())
reg.fit(X_train, c_train)
c_pred = reg.predict(X_test)

print("MAE :", round(mean_absolute_error(c_test, c_pred), 1))
print("RMSE:", round(np.sqrt(mean_squared_error(c_test, c_pred)), 1))
print("R2  :", round(r2_score(c_test, c_pred), 3))
print("negative predictions:", (c_pred < 0).sum())

### 셀 24. 치우친 비용에서 MAE와 RMSE

In [ ]:
err = (c_test - c_pred).to_numpy()
top = np.argsort(np.abs(err))[-10:]        # 오차가 가장 큰 10명
rest = np.setdiff1d(np.arange(len(err)), top)
mae = lambda e: np.abs(e).mean()
rmse = lambda e: np.sqrt((e ** 2).mean())
print("MAE  all / without top 10:",
      round(mae(err), 1), round(mae(err[rest]), 1))
print("RMSE all / without top 10:",
      round(rmse(err), 1), round(rmse(err[rest]), 1))
print("share of squared error from top 10:",
      round((err[top] ** 2).sum() / (err ** 2).sum(), 3))

## 과제 정답

**과제 1.** 진단과 처방(그리고 나이, 성별, 소득, 보험, 지역)만으로 만든 파이프라인과 모든 특성을 쓴 파이프라인의 교차검증 AUC를 비교합니다.

### 셀 25. 과제 1 정답. 특성 묶음 바꾸기

In [ ]:
comorb = ["htn", "dm", "dyslip", "ckd", "hf", "af", "cad", "stroke",
          "copd", "asthma", "depress", "dementia", "cancer",
          "liver", "oa"]
drugs = ["anticoag", "insulin", "opioid", "benzo", "antipsych",
         "diuretic"]
small = make_pipe(LogisticRegression(max_iter=1000),
                  num=["age", "female", "income_q"] + comorb + drugs,
                  cat=["insurance", "region"])
full = make_pipe(LogisticRegression(max_iter=1000))
for name, p in [("dx + drugs only", small), ("all features", full)]:
    s = cross_val_score(p, X_train, y_train, cv=skf,
                        scoring="roc_auc")
    print(f"{name:16s} CV AUC {s.mean():.3f} (SD {s.std():.3f})")

**과제 2.** k를 3, 5, 10으로 바꿔 교차검증 AUC의 평균과 겹 사이 SD를 비교합니다.

### 셀 26. 과제 2 정답. k 바꾸기

In [ ]:
for k in [3, 5, 10]:
    cv_k = StratifiedKFold(n_splits=k, shuffle=True,
                           random_state=2026)
    s = cross_val_score(make_pipe(LogisticRegression(max_iter=1000)),
                        X_train, y_train, cv=cv_k, scoring="roc_auc")
    print(f"k={k:2d}  train per fold {len(X_train) * (k - 1) // k:5d}"
          f"  mean {s.mean():.3f}  SD {s.std():.3f}"
          f"  min {s.min():.3f}  max {s.max():.3f}")

**과제 3.** 목표(cost_2023)와 치우친 특성 세 개를 log(1 + x)로 바꿔 선형회귀를 다시 맞추고, 원래 단위의 MAE, RMSE, R²와 예측 평균을 비교합니다.

### 셀 27. 과제 3 정답. 로그 변환한 목표

In [ ]:
from sklearn.compose import TransformedTargetRegressor

skewed = ["n_outpt", "los_days", "cost_2022"]
Xtr_log, Xte_log = X_train.copy(), X_test.copy()
for col in skewed:                       # 치우친 특성도 log(1 + x)로
    Xtr_log[col] = np.log1p(Xtr_log[col])
    Xte_log[col] = np.log1p(Xte_log[col])

log_model = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p, inverse_func=np.expm1)    # 목표도 log(1 + 비용)
reg_log = make_pipe(log_model)
reg_log.fit(Xtr_log, c_train)
c_pred_log = reg_log.predict(Xte_log)        # 만원 단위로 되돌린 예측

for name, pr in [("raw", c_pred), ("log", c_pred_log)]:
    print(f"{name}  MAE {mean_absolute_error(c_test, pr):5.1f}"
          f"  RMSE {np.sqrt(mean_squared_error(c_test, pr)):5.1f}"
          f"  R2 {r2_score(c_test, pr):.3f}"
          f"  mean of predictions {pr.mean():5.1f}")
print("mean of actual cost:", round(c_test.mean(), 1))